# Task 1 — Supervised Learning

02_classification.py — Task 1: supervised learning (who is likely to subscribe).


Pipeline: load -> drop leakage column -> temporal split -> preprocess
(fit on train only) -> baseline -> LR / RF / Boosting / LDA / QDA ->
evaluate on the untouched holdout -> top-500 ranked table.

In [10]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
CALL_BUDGET = 500

## 1. Load + drop leakage column

In [11]:
df = pd.read_csv("data/bank-additional-full.csv", sep=";")
df = df.drop(columns=["duration"])
# DECISION already made in 01_eda.py and justified there: duration is only
# known after the call ends, so it cannot be a predictive feature.

df["y"] = (df["y"] == "yes").astype(int)

# DECISION POINT: pdays==999 is a sentinel for "never contacted before",
# not a real day count. Leaving it raw would tell linear/LDA/QDA models
# that these clients were contacted "999 days ago". We add an explicit
# flag and keep pdays as-is for tree models (which can split around 999
# natively). This is a judgement call — an equally defensible option is
# to cap/bucket pdays instead. Be ready to explain whichever you pick.
df["was_contacted_before"] = (df["pdays"] != 999).astype(int)

numeric_cols = [
    "age", "campaign", "pdays", "previous", "was_contacted_before",
    "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m",
    "nr.employed",
]
categorical_cols = [
    "job", "marital", "education", "default", "housing", "loan",
    "contact", "month", "day_of_week", "poutcome",
]
feature_cols = numeric_cols + categorical_cols

## 2. Temporal split — NOT a random shuffle

In [12]:
# The dataset is ordered by date. We keep that order and cut it: the last
# slice in time is the holdout, exactly like the bank would only ever be
# able to evaluate a model on contacts that happen AFTER it was trained.
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx].copy()
holdout_df = df.iloc[split_idx:].copy()

print(f"Train rows:   {len(train_df)}  (positive rate "
      f"{train_df['y'].mean():.4f})")
print(f"Holdout rows: {len(holdout_df)}  (positive rate "
      f"{holdout_df['y'].mean():.4f})")
# *** REAL RESULT, NOT A BUG — CHECK THIS YOURSELF *** at an 80/20 cut,
# the positive rate jumps from ~6% in train to ~31% in the holdout. This
# is not random noise: the dataset is date-ordered and the later contacts
# were a materially different campaign period (fewer, more targeted calls
# with a much higher success rate — you can see this if you plot the
# positive rate against row index / month). This is exactly the kind of
# thing the assignment wants you to notice and explain rather than paper
# over. It means: (a) "base rate" and "lift" are holdout-period-specific,
# not a global constant, (b) a single 80/20 cut may not be the most
# defensible choice — try moving the cut point and see how much results
# change, and report that sensitivity rather than picking one split
# silently. DECISION POINT, not a default to keep without checking it.

X_train, y_train = train_df[feature_cols], train_df["y"]
X_holdout, y_holdout = holdout_df[feature_cols], holdout_df["y"]

Train rows:   32950  (positive rate 0.0637)
Holdout rows: 8238  (positive rate 0.3083)


## 3. Preprocessing — fit ONLY on training data

In [13]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         categorical_cols),
    ]
)
# "unknown" is treated as just another category level here, not imputed
# away — it may itself be informative (e.g. non-disclosure correlates
# with something). OneHotEncoder(handle_unknown="ignore") also means any
# holdout category never seen in training gets an all-zero row instead
# of crashing, without ever peeking at holdout values to decide encoding.

## 4. Models to compare

In [14]:
models = {
    "baseline_prior": DummyClassifier(strategy="prior"),
    "logistic_regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    # DECISION: class_weight="balanced" is our chosen imbalance strategy
    # (reweight the loss instead of resampling rows). Compare against the
    # unweighted version to see if it actually helps before committing.
    "logistic_regression_unweighted": LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1,
    ),  # bagging method
    "xgboost": XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        eval_metric="aucpr", random_state=RANDOM_STATE,
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    ),  # boosting method
    "lda": LinearDiscriminantAnalysis(),
    "qda": QuadraticDiscriminantAnalysis(reg_param=0.1),
    # reg_param shrinks each class's covariance estimate toward a shared,
    # better-conditioned one. Without it QDA fails outright here (one-hot
    # categoricals make the per-class covariance matrix singular) — that
    # failure IS a real, reportable result, not a bug to silently patch
    # around. We regularize just enough to let it run so we have a number
    # to compare, and we still expect it to underperform.
    # LDA/QDA are diagnostic comparisons, not expected to win. One-hot
    # encoded categoricals create near-collinear, non-Gaussian features —
    # exactly the assumptions LDA/QDA rely on. Expect them to underperform
    # and SAY SO rather than forcing them to be the final model.
}

results = {}
fitted_pipelines = {}

for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("clf", clf)])
    try:
        pipe.fit(X_train, y_train)
    except Exception as exc:
        print(f"\n[{name}] FAILED TO FIT: {exc}")
        print(f"[{name}] -> document this as a real result, don't hide it.")
        results[name] = {"roc_auc": np.nan, "pr_auc": np.nan,
                          f"precision@{CALL_BUDGET}": np.nan,
                          f"lift@{CALL_BUDGET}": np.nan}
        continue
    fitted_pipelines[name] = pipe

    if hasattr(pipe, "predict_proba"):
        proba = pipe.predict_proba(X_holdout)[:, 1]
    else:
        proba = pipe.decision_function(X_holdout)

    roc_auc = roc_auc_score(y_holdout, proba)
    pr_auc = average_precision_score(y_holdout, proba)

    ranked = pd.DataFrame({"y_true": y_holdout.values, "score": proba})
    ranked = ranked.sort_values("score", ascending=False).reset_index(drop=True)
    top_k = ranked.head(CALL_BUDGET)
    precision_at_k = top_k["y_true"].mean()
    base_rate = y_holdout.mean()
    lift_at_k = precision_at_k / base_rate

    results[name] = {
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        f"precision@{CALL_BUDGET}": precision_at_k,
        f"lift@{CALL_BUDGET}": lift_at_k,
    }

results_df = pd.DataFrame(results).T.round(4)
print("\n")
print(f"MODEL COMPARISON (holdout, base rate = {y_holdout.mean():.4f})")
print(results_df)
results_df.to_csv("outputs/model_comparison.csv")



MODEL COMPARISON (holdout, base rate = 0.3083)
                                roc_auc  pr_auc  precision@500  lift@500
baseline_prior                   0.5000  0.3083          0.064    0.2076
logistic_regression              0.7001  0.5043          0.602    1.9525
logistic_regression_unweighted   0.7483  0.5271          0.538    1.7449
random_forest                    0.7121  0.5048          0.622    2.0173
xgboost                          0.6132  0.4200          0.536    1.7384
lda                              0.6203  0.4477          0.488    1.5827
qda                              0.6891  0.5250          0.630    2.0433


## 5. Pick the final model — by the business metric, not by eyeballing AUC

In [15]:
# DECISION: for a fixed 500-call budget, precision@500 is the number that
# actually matters to the bank. We select the final model on that basis.
final_model_name = results_df[f"precision@{CALL_BUDGET}"].idxmax()
print(f"\nSelected final model: {final_model_name} "
      f"(highest precision@{CALL_BUDGET} on the holdout)")
final_pipe = fitted_pipelines[final_model_name]


Selected final model: qda (highest precision@500 on the holdout)


## 6. Full evaluation report for the final model

In [16]:
final_proba = final_pipe.predict_proba(X_holdout)[:, 1]
final_pred = (final_proba >= 0.5).astype(int)

print("\n")
print(f"CONFUSION MATRIX @ 0.5 threshold — {final_model_name}")
print(confusion_matrix(y_holdout, final_pred))
print("\nClassification report:")
print(classification_report(y_holdout, final_pred, digits=3))
# NOTE: 0.5 is an arbitrary threshold here — the real decision rule for
# this business problem is "call the top 500 by score", not a 0.5 cutoff.
# The confusion matrix above is shown for completeness, not as the
# operating point we actually recommend.



CONFUSION MATRIX @ 0.5 threshold — qda
[[4855  843]
 [1371 1169]]

Classification report:
              precision    recall  f1-score   support

           0      0.780     0.852     0.814      5698
           1      0.581     0.460     0.514      2540

    accuracy                          0.731      8238
   macro avg      0.680     0.656     0.664      8238
weighted avg      0.719     0.731     0.722      8238



## 7. Error analysis — look, don't invent

In [17]:
eval_df = holdout_df.copy()
eval_df["predicted_score"] = final_proba
eval_df["predicted_label"] = final_pred
eval_df["false_positive"] = (eval_df["predicted_label"] == 1) & (eval_df["y"] == 0)
eval_df["false_negative"] = (eval_df["predicted_label"] == 0) & (eval_df["y"] == 1)

print("\n")
print("ERROR ANALYSIS — poutcome breakdown")
print("False positive rate by prior outcome:")
print(eval_df.groupby("poutcome")["false_positive"].mean().round(3))
print("\nFalse negative rate by prior outcome:")
print(eval_df.groupby("poutcome")["false_negative"].mean().round(3))
# Read these two tables yourself and write down, in your own words, what
# pattern (if any) actually shows up — don't assume the answer in advance.



ERROR ANALYSIS — poutcome breakdown
False positive rate by prior outcome:
poutcome
failure        0.092
nonexistent    0.055
success        0.310
Name: false_positive, dtype: float64

False negative rate by prior outcome:
poutcome
failure        0.169
nonexistent    0.206
success        0.000
Name: false_negative, dtype: float64


## 8. Top-500 ranked holdout table (the required deliverable)

In [18]:
top500 = (
    eval_df[["predicted_score", "y"]]
    .sort_values("predicted_score", ascending=False)
    .head(CALL_BUDGET)
    .reset_index(drop=True)
)
top500.insert(0, "rank", np.arange(1, len(top500) + 1))
top500 = top500.rename(columns={"y": "historical_outcome"})
top500.to_csv("outputs/top_500_holdout.csv", index=False)
print("\n")
print("TOP-500 TABLE (offline demonstration on historical holdout — "
      "NOT a live call list)")
print(top500.head(10))
print(f"... saved full table to top_500_holdout.csv "
      f"({top500['historical_outcome'].sum()} actual subscribers in the "
      f"top {CALL_BUDGET})")



TOP-500 TABLE (offline demonstration on historical holdout — NOT a live call list)
   rank  predicted_score  historical_outcome
0     1              1.0                   0
1     2              1.0                   1
2     3              1.0                   1
3     4              1.0                   0
4     5              1.0                   0
5     6              1.0                   0
6     7              1.0                   1
7     8              1.0                   0
8     9              1.0                   0
9    10              1.0                   0
... saved full table to top_500_holdout.csv (315 actual subscribers in the top 500)
